# E59b — 34B 컬럼의 커버리지 구멍 메우기 + AIME 신규 렌더링

**런타임 → A100. 총 1.5~2시간.**

E59로 만든 34B 컬럼은 충실도는 이겼지만(`corr(ax31)` **0.699** vs Qwen 0.612) **커버리지가 0.753**에
그친다 — 기존 컬럼 A는 0.909다. dev 880문항 기준 구멍:

| 구멍 | 개수 | 원인 | 처방 |
|---|---:|---|---|
| dmmath 115 · gsm8k 38 · aime 12 · truthfulqa 2 | 167 | `build_pool.py`가 공개 출처에서 못 되살리는 문항 | ③ `public_all.jsonl`(공개 프롬프트 그 자체) 라벨링 |
| longdoc | 50 | 8192 길이 한도에서 생성 시 건너뜀 | ③ 한도 16384 |

**그리고 AIME**: 대회가 `aime-selection.json`으로 못박은 실제 AIME 36문항(2024:18 / 2025:18)을 우리 컬럼은
**0개** 커버한다. 그런데 이 36문항의 주최측 실측치는

| 모델 | 평균 점수 | 출력 중앙값 |
|---|---:|---:|
| `ax31-light` | 0.069 | 967 |
| `ax31` | 0.132 | 907 |
| **`axk1-think`** | **0.792** | 9,765 |

즉 **업그레이드 가치가 가장 큰 family인데 prior가 눈을 감고 있다.** `build_pool.py`에는 AIME 렌더러가
아예 없어서 공개셋 밖 AIME 문항은 영원히 미커버다. ③b가 이를 메우며, 출처는 해시로 검증했다 —
대회 36문항 중 **35개를 글자 단위로 재현**한다.

AIME만 `--max-tokens`를 2048로 올린다. 384에서는 모든 AIME 문항이 상한에 붙어 **길이 특징의 정보량이
0이 되기 때문**이다(주최측 `ax31` 중앙값 907).

In [ ]:
#@title ① 번들 + 이전 라벨 복원  (e59b_colab_bundle.zip 과 e59_resume.zip 을 MyDrive 에 둘 것)
import os, zipfile, glob, shutil

def fetch(name):
    """MyDrive 에서 가져오되, 잘린 파일을 조용히 넘기지 않는다."""
    if not os.path.exists(name):
        c = glob.glob('/content/drive/MyDrive/**/' + name, recursive=True)
        if not c:
            return None
        shutil.copyfile(c[0], name)          # os.system('cp') 은 실패해도 조용하다
    size = os.path.getsize(name)
    if not zipfile.is_zipfile(name):
        head = open(name, 'rb').read(4)
        print(f'*** {name} 이 zip 이 아니다 ({size:,} bytes, 앞 4바이트 {head!r}) -> 지우고 다시 올릴 것 ***')
        return None
    print(f'{name}: OK ({size/1e6:.1f} MB)')
    return name

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('drive skip:', e)

if fetch('e59b_colab_bundle.zip') is None:
    from google.colab import files; files.upload()
with zipfile.ZipFile('e59b_colab_bundle.zip') as z: z.extractall('.')

# 재개용 라벨: 새 이름(e59_resume.zip, 7 MB) 우선, 없으면 예전 이름도 시도
resume = fetch('e59_resume.zip') or fetch('e59_out.zip')
if resume:
    with zipfile.ZipFile(resume) as z: z.extractall('/content/official-router')
    print('이전 라벨 복원됨 — 이미 라벨링한 문항은 자동으로 건너뛴다')
else:
    print('*** 재개 파일이 없다: ③이 36,960건을 처음부터 다시 돌게 된다. 계속하지 말고 알릴 것 ***')

%cd /content/official-router
!grep -c has_gold colab-label/run_labels.py
print('^ 1 이상이어야 gold 없는 문항(dmmath·longdoc)에서 안 죽는다')
!wc -l colab-label/out/*.jsonl 2>/dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title ② 의존성 + 설정
!pip -q install vllm datasets bitsandbytes 2>&1 | tail -2
!pip -q uninstall -y torchaudio 2>&1 | tail -1
import os, torch, transformers, vllm
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
os.environ.update(MODEL='skt/A.X-3.1', N='4', TEMP='0.7', INSTR='v1',
                  MAXTOK='384',       # 일반 문항
                  AIMETOK='2048',     # AIME: 주최측 ax31 출력 중앙값 907
                  MAXLEN='16384',     # E59의 8192 에서 상향: 장문 문항 회수
                  QUANT='' if GB >= 78 else 'bitsandbytes',
                  UTIL='0.95' if GB >= 78 else '0.90',
                  TOKENIZERS_PARALLELISM='false')
print('OK', torch.__version__, transformers.__version__, f'{GB:.0f} GB',
      'quant=' + (os.environ['QUANT'] or 'bf16'))

In [ ]:
#@title ③ 공개 2,640 프롬프트 라벨링 (~1시간) — gold 없는 dmmath/longdoc 도 이제 살아남는다
import os
!PYTHONPATH=src python -X utf8 colab-label/build_public_all.py
!cp colab-label/bundle/public_all.jsonl colab-label/bundle/pool.jsonl
!wc -l colab-label/bundle/pool.jsonl
LAB = 'colab-label/out/labels_pool_T0.7.jsonl'
before = sum(1 for _ in open(LAB, encoding='utf-8')) if os.path.exists(LAB) else 0
print('라벨링 전 행 수:', before)
!PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pool \
    --model "$MODEL" --engine vllm --n $N --temp $TEMP --instruct "$INSTR" \
    --max-model-len $MAXLEN --max-tokens $MAXTOK --gpu-util $UTIL \
    ${QUANT:+--quant $QUANT} \
    --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -25
after = sum(1 for _ in open(LAB, encoding='utf-8'))
print('=== 새로 추가된 행:', after - before, ' (기대: 2,400~2,600) ===')
if after == before:
    print('*** 0건이면 위 오류 메시지를 그대로 복사해서 알릴 것 ***')

In [ ]:
#@title ③b AIME 신규 문항 렌더링 + 라벨링 (130문항, 출력이 길어 ~30분)
# 먼저 출처 검증: 대회 36문항 중 몇 개를 글자 단위로 재현하는지 (기대 35/36)
!PYTHONPATH=src python -X utf8 colab-label/build_pool_aime.py \
    --out colab-label/bundle/aime.jsonl \
    --have-digests colab-label/covered_digests.txt 2>&1 | tail -12
!cp colab-label/bundle/aime.jsonl colab-label/bundle/pool.jsonl
!PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pool \
    --model "$MODEL" --engine vllm --n $N --temp $TEMP --instruct "$INSTR" \
    --max-model-len $MAXLEN --max-tokens $AIMETOK --gpu-util $UTIL \
    ${QUANT:+--quant $QUANT} --tag _aime \
    --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -20
!wc -l colab-label/out/labels_pool_T0.7_aime.jsonl

In [ ]:
#@title ④ 커버리지 확인 — dev 0.90 이상이면 성공, 대회 AIME 36/36 이면 완전 커버
import os, json, hashlib, sys
from pathlib import Path
!PYTHONPATH=src python -X utf8 colab-label/to_prior_labels.py \
    colab-label/out/labels_pool_T0.7.jsonl colab-label/out/labels_mid_pool.jsonl --model ax31-34b-v1
if os.path.exists('colab-label/out/labels_pool_T0.7_aime.jsonl'):
    !PYTHONPATH=src python -X utf8 colab-label/to_prior_labels.py \
        colab-label/out/labels_pool_T0.7_aime.jsonl colab-label/out/labels_mid_aime.jsonl --model ax31-34b-v1

sys.path.insert(0, 'src')
from ossp_router.heuristic import episode_text
from ossp_router.protocol import load_input

# E59 에서 이미 커버한 38,833 항목은 다이제스트 목록으로 들고 온다 (거대한 bundle/*.jsonl 없이 정확)
ent = set()
cov = 'colab-label/covered_digests.txt'
if os.path.exists(cov):
    ent |= {l.strip() for l in open(cov, encoding='utf-8') if l.strip()}
    print('E59 시점 커버 다이제스트:', len(ent))

# 이번에 새로 라벨링한 것들 (public_all + aime) 을 더한다
prompts = {}
for p in ['public_all', 'aime', 'all', 'ext']:
    f = 'colab-label/bundle/' + p + '.jsonl'
    if not os.path.exists(f): continue
    for line in open(f, encoding='utf-8'):
        if line.strip():
            r = json.loads(line); prompts[r['id']] = r['prompt']
for p in ['labels_mid_pool', 'labels_mid_aime', 'labels_gate']:
    f = 'colab-label/out/' + p + '.jsonl'
    if not os.path.exists(f): continue
    for line in open(f, encoding='utf-8'):
        if line.strip():
            r = json.loads(line); t = prompts.get(r['id'])
            if t: ent.add(hashlib.sha256(t.encode()).hexdigest())
print('새 컬럼 엔트리 합계:', len(ent))

for split in ('train', 'dev'):
    eps = list(load_input(Path('data/materialized/' + split + '/inputs.json')).episodes)
    hit = sum(hashlib.sha256(episode_text(e).encode()).hexdigest() in ent for e in eps)
    print('  %s 커버리지 %d/%d = %.3f   (목표 0.909, E59 시점 0.753)' % (split, hit, len(eps), hit / len(eps)))

sel = set()
for s in ('train', 'dev'):
    for e in json.load(open('data/' + s + '/aime-selection.json', encoding='utf-8'))['episodes']:
        sel.add(e['episode_id'])
hit = tot = 0
for s in ('train', 'dev'):
    for e in load_input(Path('data/materialized/' + s + '/inputs.json')).episodes:
        if e.episode_id in sel:
            tot += 1; hit += hashlib.sha256(episode_text(e).encode()).hexdigest() in ent
print('  대회 AIME %d/%d   (E59 시점 0/36)' % (hit, tot))

In [ ]:
#@title ⑤ 결과 회수 → MyDrive  (라벨만 담으므로 작다)
!cd /content/official-router && zip -qr /content/e59b_out.zip \
    colab-label/out/labels_pool_T0.7.jsonl colab-label/out/labels_pool_T0.7_aime.jsonl \
    colab-label/out/labels_mid_pool.jsonl colab-label/out/labels_mid_aime.jsonl \
    colab-label/out/labels_gate.jsonl \
    colab-label/bundle/public_all.jsonl colab-label/bundle/aime.jsonl
!ls -la /content/e59b_out.zip
import zipfile
print('zip 정상:', zipfile.is_zipfile('/content/e59b_out.zip'))
!cp /content/e59b_out.zip /content/drive/MyDrive/
import os, shutil
src, dst = '/content/e59b_out.zip', '/content/drive/MyDrive/e59b_out.zip'
print('MyDrive 사본 크기 일치:', os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src))
print('   (일치하지 않으면 왼쪽 파일탐색기 📁 -> content -> e59b_out.zip 우클릭 -> 다운로드)')